# Activité Pratique N°4 — Architecture Transformer
## Partie 2 — Exercice 2 : Mini-GPT

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

Construire un Transformer autorégressif qui prédit le caractère suivant.

**Objectif :** un Transformer **autorégressif** entraîné depuis zéro qui prédit le **caractère suivant** (tokenisation par caractères).
Le corpus est `corpus.txt` (quelques pages de texte en français sur l'IA, écrites pour cet exercice, dans le dossier de ce notebook).

In [ ]:
import copy
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F


def fixer_graine(graine=42):
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)


def nb_parametres(modele):
    return sum(p.numel() for p in modele.parameters())


fixer_graine()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| appareil :", device)

## Données et vocabulaire

In [ ]:
texte = Path("corpus.txt").read_text(encoding="utf-8")
caracteres = sorted(set(texte))
stoi = {c: i for i, c in enumerate(caracteres)}
itos = {i: c for c, i in stoi.items()}
taille_vocabulaire = len(caracteres)

tokens = torch.tensor([stoi[c] for c in texte], dtype=torch.long)
coupe = int(0.9 * len(tokens))
tokens_train, tokens_val = tokens[:coupe], tokens[coupe:]
print(f"{len(texte):,} caractères | vocabulaire : {taille_vocabulaire} caractères")
print(f"train : {len(tokens_train):,} | validation : {len(tokens_val):,}")

# Exemple d'une paire entrée/cible
longueur = 64
x = tokens[:longueur]
y = tokens[1:longueur + 1]
print("\nEntrée :", repr("".join(itos[int(i)] for i in x[:30])))
print("Cible  :", repr("".join(itos[int(i)] for i in y[:30])), "<- décalée d'un caractère")


def lot(donnees, batch=64):
    ix = torch.randint(len(donnees) - longueur - 1, (batch,))
    xb = torch.stack([donnees[i:i + longueur] for i in ix])
    yb = torch.stack([donnees[i + 1:i + longueur + 1] for i in ix])
    return xb.to(device), yb.to(device)

## Modèle

**Bloc GPT** (normalisation *avant* chaque sous-couche) :

```
x ─► LayerNorm ─► Attention multi-têtes causale ─► (+) ─► LayerNorm ─► MLP 64→256→64 (GELU) ─► (+) ─►
└─────────────────────────────────────────────────┘ └──────────────────────────────────────┘
```
Masque causal à utiliser avec `torch.nn.MultiheadAttention` : `True` signifie « relation d'attention interdite ».

In [ ]:
class BlocGPT(nn.Module):
    def __init__(self, d_model=64, nhead=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attention = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 256), nn.GELU(), nn.Linear(256, d_model), nn.Dropout(dropout))

    def forward(self, x):
        T = x.shape[1]
        masque = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)
        h = self.ln1(x)
        a, _ = self.attention(h, h, h, attn_mask=masque, need_weights=False)
        x = x + a                       # 1re connexion résiduelle
        x = x + self.mlp(self.ln2(x))   # 2e connexion résiduelle
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocabulaire, d_model=64, nhead=4, blocs=2, longueur_max=64):
        super().__init__()
        self.longueur_max = longueur_max
        self.tokens = nn.Embedding(vocabulaire, d_model)
        self.positions = nn.Embedding(longueur_max, d_model)
        self.blocs = nn.Sequential(*[BlocGPT(d_model, nhead) for _ in range(blocs)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocabulaire)

    def forward(self, idx):                                   # (B, T)
        T = idx.shape[1]
        x = self.tokens(idx) + self.positions(torch.arange(T, device=idx.device))  # (B, T, 64)
        return self.tete(self.ln_final(self.blocs(x)))         # (B, T, vocabulaire)

    @torch.no_grad()
    def generer(self, amorce, n=200, temperature=1.0):
        self.eval()
        idx = torch.tensor([[stoi[c] for c in amorce]], device=next(self.parameters()).device)
        for _ in range(n):
            logits = self(idx[:, -self.longueur_max:])[:, -1, :] / temperature
            suivant = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, suivant], dim=1)
        return "".join(itos[int(i)] for i in idx[0])


fixer_graine()
gpt = MiniGPT(taille_vocabulaire).to(device)
xb, yb = lot(tokens_train)
print("Identifiants en entrée :", tuple(xb.shape))
print("Représentations        :", tuple((gpt.tokens(xb) + gpt.positions(torch.arange(64, device=device))).shape))
print("Logits                 :", tuple(gpt(xb).shape))
print("Cibles                 :", tuple(yb.shape))
print("Paramètres             :", nb_parametres(gpt))

## Entraînement (entropie croisée sur toutes les positions)

In [ ]:
@torch.no_grad()
def perte_moyenne(modele, donnees, iterations=20):
    modele.eval()
    pertes = []
    for _ in range(iterations):
        xb, yb = lot(donnees)
        pertes.append(F.cross_entropy(modele(xb).reshape(-1, taille_vocabulaire), yb.reshape(-1)).item())
    modele.train()
    return sum(pertes) / len(pertes)


fixer_graine()
gpt = MiniGPT(taille_vocabulaire).to(device)
optimiseur = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
debut = time.time()
gpt.train()
for etape in range(1, 3001):
    xb, yb = lot(tokens_train)
    perte = F.cross_entropy(gpt(xb).reshape(-1, taille_vocabulaire), yb.reshape(-1))
    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()
    if etape == 1 or etape % 250 == 0:
        print(f"Étape {etape:4d} | perte train {perte_moyenne(gpt, tokens_train):.3f} "
              f"| perte validation {perte_moyenne(gpt, tokens_val):.3f}")
print(f"\n3000 étapes en {time.time() - debut:.0f} s "
      f"(référence : un modèle qui tire au hasard a une perte de ln({taille_vocabulaire}) = {math.log(taille_vocabulaire):.2f})")

## Vérification essentielle : la causalité

En mode `eval()`, modifier un caractère **futur** ne doit pas modifier les logits des positions **précédentes**.

In [ ]:
gpt.eval()
xb, _ = lot(tokens_val, batch=4)
position = 40
xb_modifie = xb.clone()
xb_modifie[:, position] = (xb[:, position] + 1) % taille_vocabulaire
with torch.no_grad():
    l1, l2 = gpt(xb), gpt(xb_modifie)
avant = torch.allclose(l1[:, :position], l2[:, :position], atol=1e-6)
apres = not torch.allclose(l1[:, position:], l2[:, position:], atol=1e-6)
print(f"Positions 0 à {position - 1} inchangées : {avant}")
print(f"Positions {position} à 63 modifiées    : {apres}")
assert avant and apres

## Génération de 200 caractères à différentes températures

La température divise les logits avant le softmax : $p_i \propto \exp(z_i / T)$.

In [ ]:
for temperature in [0.5, 1.0, 1.5]:
    torch.manual_seed(0)
    print(f"=== Température {temperature} ===")
    print(gpt.generer("Le Transformer ", n=200, temperature=temperature), "\n")

## Questions

**Que se passe-t-il pendant l'entraînement si le masque causal est supprimé ?**
Chaque position pourrait regarder la position suivante, qui contient justement la réponse (la cible est l'entrée décalée d'un caractère).
Le modèle apprendrait à **recopier** : la perte d'entraînement chuterait presque à zéro, mais le modèle serait inutilisable en génération, où le futur n'existe pas encore.

**Quel effet la température a-t-elle sur la génération ?**
- $T = 0.5$ : distribution plus pointue, caractères les plus probables presque toujours choisis → texte régulier mais répétitif, souvent recopié du corpus.
- $T = 1$ : échantillonnage selon les probabilités apprises.
- $T = 1.5$ : distribution aplatie, caractères rares plus souvent tirés → texte varié mais plein de mots inventés.

**Pourquoi décaler les cibles d'une position ?**
Le modèle apprend $p(c_{t+1} \mid c_1,\dots,c_t)$ : la cible de la position $t$ est le caractère $t+1$.
Une séquence de 64 caractères fournit ainsi 64 exemples à la fois, et c'est exactement la tâche répétée pendant la génération.